# ByteTrack Evaluation Notebook

Notebook này nằm ngoài thư mục `web` và dùng để đánh giá khả năng hoạt động của ByteTrack trong hệ thống nhận diện phương tiện.

Dữ liệu mặc định: `D:/2026/DATN/Data/dataset_aug`.

Lưu ý quan trọng: `dataset_aug` hiện là dataset YOLO gồm ảnh + bbox theo frame, không có `object_id` thật cho từng xe. Vì vậy notebook này hỗ trợ 2 mức đánh giá:

- Đánh giá trực tiếp được: `FPS`, `MAE/RMSE` số lượng xe so với label YOLO.
- Đánh giá tracking proxy: tạo `pseudo_id` bằng cách nối bbox ground truth giữa các frame liên tiếp theo IoU, sau đó tính `MOTA/IDF1` xấp xỉ.

Nếu muốn `MOTA/IDF1` chuẩn theo MOTChallenge/TrackEval, cần annotation có ID thật: `frame_id, object_id, bbox, class` cho từng frame.


## 1. Cấu hình môi trường

Cell này tự tìm project root, import config/backend hiện tại và đọc model/config ByteTrack từ `.env` của backend.


In [ ]:
from __future__ import annotations

import os
import re
import sys
import time
import math
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'web').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'web').exists():
    PROJECT_ROOT = Path('D:/2026/DATN/code').resolve()

BACKEND_DIR = PROJECT_ROOT / 'web' / 'backend'
DATASET_AUG_DIR = Path('D:/2026/DATN/Data/dataset_aug')
VIDEO_DIR = Path('D:/2026/DATN/Data/video')
OUTPUT_DIR = PROJECT_ROOT / 'evaluation' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)  # app.core.config loads backend/.env relative to cwd

from app.core.config import settings
from app.services.frame_utils import resize_to_max
from app.services.tracking import (
    ByteTrackWrapper,
    detections_to_array,
    repair_tracked_detections,
    tracks_to_detections,
)

CLASS_NAMES = list(settings.class_names)

def resolve_model_path(raw_path: str | os.PathLike[str]) -> Path:
    path = Path(raw_path)
    if not path.is_absolute():
        path = BACKEND_DIR / path
    return path.resolve()

# Có thể override bằng biến môi trường EVAL_MODEL_PATH nếu muốn benchmark .pt/.onnx/.engine khác.
MODEL_PATH = resolve_model_path(os.getenv('EVAL_MODEL_PATH', settings.model_path))

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('BACKEND_DIR       =', BACKEND_DIR)
print('DATASET_AUG_DIR  =', DATASET_AUG_DIR, '| exists =', DATASET_AUG_DIR.exists())
print('VIDEO_DIR         =', VIDEO_DIR, '| exists =', VIDEO_DIR.exists())
print('MODEL_PATH        =', MODEL_PATH, '| exists =', MODEL_PATH.exists())
print('CLASS_NAMES       =', CLASS_NAMES)
print('ByteTrack config  =', {
    'enabled': settings.bytetrack_enabled,
    'track_thresh': settings.bytetrack_conf_high,
    'match_thresh': settings.bytetrack_iou_high,
    'track_buffer': settings.bytetrack_track_buffer,
    'min_box_area': settings.bytetrack_min_box_area,
    'frame_skip': settings.bytetrack_frame_skip,
    'repair_enabled': settings.bytetrack_repair_enabled,
    'repair_iou': settings.bytetrack_repair_iou,
})


## 2. Đọc dataset YOLO theo chuỗi frame

`dataset_aug` có cả ảnh gốc và ảnh augment. Với tracking, chỉ dùng ảnh gốc theo thứ tự `videoX_frame_XXXXX.jpg`; ảnh `_augXXXX` bị bỏ qua vì không còn là chuỗi thời gian tự nhiên.


In [ ]:
FRAME_RE = re.compile(r'^(?P<video>video\d+)_frame_(?P<frame>\d+)(?P<aug>_aug\d+)?\.(jpg|jpeg|png)$', re.IGNORECASE)

def discover_sequences(dataset_dir: Path, include_aug: bool = False) -> dict[str, list[dict]]:
    image_dir = dataset_dir / 'images'
    sequences: dict[str, list[dict]] = defaultdict(list)
    for image_path in sorted(image_dir.glob('*')):
        match = FRAME_RE.match(image_path.name)
        if not match:
            continue
        if match.group('aug') and not include_aug:
            continue
        video_name = match.group('video')
        frame_idx = int(match.group('frame'))
        label_path = dataset_dir / 'labels' / f'{image_path.stem}.txt'
        sequences[video_name].append({
            'video': video_name,
            'frame_idx': frame_idx,
            'image_path': image_path,
            'label_path': label_path,
        })
    for rows in sequences.values():
        rows.sort(key=lambda row: row['frame_idx'])
    return dict(sequences)

sequences = discover_sequences(DATASET_AUG_DIR, include_aug=False)
sequence_summary = pd.DataFrame([
    {
        'sequence': name,
        'frames': len(rows),
        'first_frame': rows[0]['frame_idx'] if rows else None,
        'last_frame': rows[-1]['frame_idx'] if rows else None,
        'missing_labels': sum(1 for row in rows if not row['label_path'].exists()),
    }
    for name, rows in sorted(sequences.items())
])
sequence_summary


In [ ]:
def yolo_xywh_to_xyxy(cx: float, cy: float, bw: float, bh: float, width: int, height: int) -> list[float]:
    x1 = (cx - bw / 2.0) * width
    y1 = (cy - bh / 2.0) * height
    x2 = (cx + bw / 2.0) * width
    y2 = (cy + bh / 2.0) * height
    return [
        max(0.0, min(float(x1), float(width - 1))),
        max(0.0, min(float(y1), float(height - 1))),
        max(0.0, min(float(x2), float(width - 1))),
        max(0.0, min(float(y2), float(height - 1))),
    ]

def read_yolo_labels(label_path: Path, width: int, height: int) -> list[dict]:
    boxes: list[dict] = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        class_id = int(float(parts[0]))
        cx, cy, bw, bh = map(float, parts[1:5])
        x1, y1, x2, y2 = yolo_xywh_to_xyxy(cx, cy, bw, bh, width, height)
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append({
            'class_id': class_id,
            'class_name': CLASS_NAMES[class_id] if 0 <= class_id < len(CLASS_NAMES) else str(class_id),
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
        })
    return boxes

def choose_sequence(sequences: dict[str, list[dict]], sequence_name: str | None = None) -> str:
    if not sequences:
        raise ValueError(f'No valid sequences found in {DATASET_AUG_DIR}')
    if sequence_name:
        if sequence_name not in sequences:
            raise ValueError(f'Unknown sequence {sequence_name!r}. Available: {sorted(sequences)}')
        return sequence_name
    return max(sequences, key=lambda name: len(sequences[name]))

def load_sequence_records(sequence_name: str | None = None, max_frames: int | None = 300) -> list[dict]:
    selected = choose_sequence(sequences, sequence_name)
    rows = sequences[selected]
    if max_frames is not None:
        rows = rows[:max_frames]
    records: list[dict] = []
    for row in rows:
        image = cv2.imread(str(row['image_path']))
        if image is None:
            continue
        height, width = image.shape[:2]
        gt = read_yolo_labels(row['label_path'], width, height)
        records.append({
            **row,
            'width': width,
            'height': height,
            'gt': gt,
        })
    print(f'Loaded {len(records)} frames from sequence {selected!r}')
    return records


## 3. Tạo pseudo ground truth ID

Vì label YOLO không có ID thật, cell này tạo `pseudo_id` bằng cách nối bbox cùng class giữa các frame gần nhau nếu IoU đủ cao.

Đây không phải ground truth tracking chuẩn, nhưng đủ hữu ích để phát hiện các vấn đề như track ID nhảy liên tục, mất track hoặc tạo quá nhiều ID mới.


In [ ]:
def bbox_iou_xyxy(a: dict, b: dict) -> float:
    x1 = max(float(a['x1']), float(b['x1']))
    y1 = max(float(a['y1']), float(b['y1']))
    x2 = min(float(a['x2']), float(b['x2']))
    y2 = min(float(a['y2']), float(b['y2']))
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    area_a = max(0.0, float(a['x2']) - float(a['x1'])) * max(0.0, float(a['y2']) - float(a['y1']))
    area_b = max(0.0, float(b['x2']) - float(b['x1'])) * max(0.0, float(b['y2']) - float(b['y1']))
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def build_pseudo_gt(records: list[dict], iou_thr: float = 0.30, max_lost: int = 2) -> list[dict]:
    next_id = 1
    active: list[dict] = []
    output: list[dict] = []

    for rec in records:
        frame_idx = int(rec['frame_idx'])
        current = [dict(box) for box in rec['gt']]
        usable_active = [track for track in active if frame_idx - int(track['last_frame']) <= max_lost + 1]

        candidates: list[tuple[float, int, int]] = []
        for active_idx, track in enumerate(usable_active):
            for cur_idx, box in enumerate(current):
                if int(track['class_id']) != int(box['class_id']):
                    continue
                iou = bbox_iou_xyxy(track, box)
                if iou >= iou_thr:
                    candidates.append((iou, active_idx, cur_idx))
        candidates.sort(reverse=True, key=lambda item: item[0])

        used_active: set[int] = set()
        used_current: set[int] = set()
        for _iou, active_idx, cur_idx in candidates:
            if active_idx in used_active or cur_idx in used_current:
                continue
            current[cur_idx]['pseudo_id'] = int(usable_active[active_idx]['pseudo_id'])
            used_active.add(active_idx)
            used_current.add(cur_idx)

        for box in current:
            if 'pseudo_id' not in box:
                box['pseudo_id'] = next_id
                next_id += 1

        active_by_id = {
            int(track['pseudo_id']): track
            for track in usable_active
            if frame_idx - int(track['last_frame']) <= max_lost
        }
        for box in current:
            active_by_id[int(box['pseudo_id'])] = {
                **box,
                'last_frame': frame_idx,
            }
        active = list(active_by_id.values())

        output.append({
            **rec,
            'gt_pseudo': current,
        })

    return output

def scale_boxes(boxes: list[dict], sx: float, sy: float) -> list[dict]:
    scaled = []
    for box in boxes:
        scaled.append({
            **box,
            'x1': float(box['x1']) * sx,
            'y1': float(box['y1']) * sy,
            'x2': float(box['x2']) * sx,
            'y2': float(box['y2']) * sy,
        })
    return scaled


## 4. Chạy YOLO + ByteTrack theo cấu hình hệ thống

Cell này dùng model từ `APP_MODEL_PATH` trong backend `.env`. Nếu muốn test model khác, set biến môi trường `EVAL_MODEL_PATH` trước khi chạy notebook.


In [ ]:
from ultralytics import YOLO

try:
    import torch
except Exception:
    torch = None

def resolve_device() -> str:
    if settings.device != 'auto':
        return settings.device
    if torch is not None and torch.cuda.is_available():
        return 'cuda:0'
    return 'cpu'

DEVICE = resolve_device()

def class_name_from_result(result, class_id: int) -> str:
    if 0 <= class_id < len(CLASS_NAMES):
        return CLASS_NAMES[class_id]
    names = getattr(result, 'names', None)
    if isinstance(names, dict) and class_id in names:
        return str(names[class_id])
    if isinstance(names, list) and 0 <= class_id < len(names):
        return str(names[class_id])
    return str(class_id)

def detect_frame(model: YOLO, frame: np.ndarray) -> tuple[list[dict], float]:
    kwargs = {
        'source': frame,
        'conf': settings.conf,
        'iou': settings.iou,
        'imgsz': settings.img_size,
        'device': DEVICE,
        'verbose': False,
    }
    if MODEL_PATH.suffix.lower() in {'.pt', '.pth'} and settings.use_half and DEVICE.startswith('cuda'):
        kwargs['half'] = True

    start = time.perf_counter()
    results = model.predict(**kwargs)
    infer_ms = (time.perf_counter() - start) * 1000.0

    detections: list[dict] = []
    if not results:
        return detections, infer_ms
    result = results[0]
    for idx, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].tolist()
        class_id = int(box.cls[0].item())
        detections.append({
            'object_id': idx,
            'class_id': class_id,
            'class_name': class_name_from_result(result, class_id),
            'confidence': float(box.conf[0].item()),
            'x1': int(xyxy[0]),
            'y1': int(xyxy[1]),
            'x2': int(xyxy[2]),
            'y2': int(xyxy[3]),
        })
    return detections, infer_ms

def run_yolo_bytetrack(records: list[dict], resize_max_dim: int | None = None, frame_rate: int = 30):
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'Model not found: {MODEL_PATH}')
    if not settings.bytetrack_enabled:
        raise RuntimeError('APP_BYTETRACK_ENABLED=false; enable it before evaluation.')

    model = YOLO(str(MODEL_PATH))
    tracker = ByteTrackWrapper(frame_rate=frame_rate)
    last_tracked: list[dict] | None = None
    track_stride = max(1, int(settings.bytetrack_frame_skip))

    gt_by_frame: dict[int, list[dict]] = {}
    pred_by_frame: dict[int, list[dict]] = {}
    timing_rows: list[dict] = []
    prediction_rows: list[dict] = []

    for pos, rec in enumerate(records, start=1):
        frame_start = time.perf_counter()
        frame = cv2.imread(str(rec['image_path']))
        if frame is None:
            continue

        if resize_max_dim:
            processed = resize_to_max(frame, resize_max_dim)
        else:
            processed = frame

        src_h, src_w = frame.shape[:2]
        dst_h, dst_w = processed.shape[:2]
        sx = dst_w / max(src_w, 1)
        sy = dst_h / max(src_h, 1)
        gt_scaled = scale_boxes(rec['gt_pseudo'], sx, sy)

        detections, infer_ms = detect_frame(model, processed)

        track_start = time.perf_counter()
        run_tracker = pos % track_stride == 0
        if run_tracker:
            tracked = tracks_to_detections(
                tracker.update(detections_to_array(detections), processed),
                CLASS_NAMES,
            )
            if settings.bytetrack_repair_enabled:
                tracked = repair_tracked_detections(
                    detections,
                    tracked,
                    last_tracked,
                    settings.bytetrack_repair_iou,
                )
            last_tracked = tracked
        else:
            tracked = last_tracked or detections
        track_ms = (time.perf_counter() - track_start) * 1000.0
        total_ms = (time.perf_counter() - frame_start) * 1000.0

        frame_idx = int(rec['frame_idx'])
        gt_by_frame[frame_idx] = gt_scaled
        pred_by_frame[frame_idx] = tracked

        timing_rows.append({
            'frame_idx': frame_idx,
            'infer_ms': infer_ms,
            'track_ms': track_ms,
            'total_ms': total_ms,
            'gt_count': len(gt_scaled),
            'pred_count': len(tracked),
        })

        for pred in tracked:
            prediction_rows.append({
                'frame_idx': frame_idx,
                'track_id': pred.get('track_id', pred.get('object_id')),
                'class_id': pred.get('class_id'),
                'class_name': pred.get('class_name'),
                'confidence': pred.get('confidence'),
                'x1': pred.get('x1'),
                'y1': pred.get('y1'),
                'x2': pred.get('x2'),
                'y2': pred.get('y2'),
            })

        if pos % 50 == 0:
            print(f'Processed {pos}/{len(records)} frames')

    timing_df = pd.DataFrame(timing_rows)
    predictions_df = pd.DataFrame(prediction_rows)
    return gt_by_frame, pred_by_frame, timing_df, predictions_df


## 5. Tính metric

Các metric chính:

- `MAE/RMSE`: sai số số lượng object mỗi frame.
- `MOTA_proxy`: dùng pseudo GT ID, công thức `1 - (FN + FP + IDSW) / GT`.
- `IDF1_proxy`: xấp xỉ bằng ghép cặp global giữa `pseudo_id` và `track_id`.
- `FPS`: số frame xử lý / tổng thời gian xử lý.


In [ ]:
def mean_absolute_error(values: list[float]) -> float:
    return float(np.mean(np.abs(values))) if values else float('nan')

def root_mean_squared_error(values: list[float]) -> float:
    return float(np.sqrt(np.mean(np.square(values)))) if values else float('nan')

def match_boxes(gt_boxes: list[dict], pred_boxes: list[dict], iou_thr: float = 0.50, class_aware: bool = True):
    candidates: list[tuple[float, int, int]] = []
    for gi, gt in enumerate(gt_boxes):
        for pi, pred in enumerate(pred_boxes):
            if class_aware and int(gt.get('class_id', -1)) != int(pred.get('class_id', -2)):
                continue
            iou = bbox_iou_xyxy(gt, pred)
            if iou >= iou_thr:
                candidates.append((iou, gi, pi))
    candidates.sort(reverse=True, key=lambda item: item[0])

    matches = []
    used_gt: set[int] = set()
    used_pred: set[int] = set()
    for iou, gi, pi in candidates:
        if gi in used_gt or pi in used_pred:
            continue
        used_gt.add(gi)
        used_pred.add(pi)
        matches.append((gi, pi, iou))

    unmatched_gt = [idx for idx in range(len(gt_boxes)) if idx not in used_gt]
    unmatched_pred = [idx for idx in range(len(pred_boxes)) if idx not in used_pred]
    return matches, unmatched_gt, unmatched_pred

def max_identity_matches(pair_counts: Counter) -> int:
    if not pair_counts:
        return 0
    gt_ids = sorted({gt_id for gt_id, _pred_id in pair_counts})
    pred_ids = sorted({pred_id for _gt_id, pred_id in pair_counts})
    gt_index = {gt_id: idx for idx, gt_id in enumerate(gt_ids)}
    pred_index = {pred_id: idx for idx, pred_id in enumerate(pred_ids)}
    matrix = np.zeros((len(gt_ids), len(pred_ids)), dtype=np.int32)
    for (gt_id, pred_id), count in pair_counts.items():
        matrix[gt_index[gt_id], pred_index[pred_id]] = int(count)

    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-matrix)
        return int(matrix[rows, cols].sum())
    except Exception:
        total = 0
        used_gt: set[int] = set()
        used_pred: set[int] = set()
        for (gt_id, pred_id), count in pair_counts.most_common():
            if gt_id in used_gt or pred_id in used_pred:
                continue
            used_gt.add(gt_id)
            used_pred.add(pred_id)
            total += int(count)
        return total

def evaluate_tracking(gt_by_frame: dict[int, list[dict]], pred_by_frame: dict[int, list[dict]], timing_df: pd.DataFrame, iou_thr: float = 0.50):
    tp = fp = fn = idsw = 0
    total_gt = 0
    total_pred = 0
    count_errors: list[int] = []
    pair_counts: Counter = Counter()
    last_pred_for_gt: dict[int, int] = {}
    per_class_errors: dict[str, list[int]] = defaultdict(list)

    for frame_idx in sorted(gt_by_frame):
        gt_boxes = gt_by_frame.get(frame_idx, [])
        pred_boxes = pred_by_frame.get(frame_idx, [])
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        count_errors.append(len(pred_boxes) - len(gt_boxes))

        gt_counts = Counter(box.get('class_name', str(box.get('class_id'))) for box in gt_boxes)
        pred_counts = Counter(box.get('class_name', str(box.get('class_id'))) for box in pred_boxes)
        for class_name in CLASS_NAMES:
            per_class_errors[class_name].append(pred_counts[class_name] - gt_counts[class_name])

        matches, unmatched_gt, unmatched_pred = match_boxes(gt_boxes, pred_boxes, iou_thr=iou_thr, class_aware=True)
        tp += len(matches)
        fn += len(unmatched_gt)
        fp += len(unmatched_pred)

        for gi, pi, _iou in matches:
            gt_id = int(gt_boxes[gi]['pseudo_id'])
            pred_id = int(pred_boxes[pi].get('track_id', pred_boxes[pi].get('object_id', pi)))
            previous_pred = last_pred_for_gt.get(gt_id)
            if previous_pred is not None and previous_pred != pred_id:
                idsw += 1
            last_pred_for_gt[gt_id] = pred_id
            pair_counts[(gt_id, pred_id)] += 1

    idtp = max_identity_matches(pair_counts)
    idfp = max(0, total_pred - idtp)
    idfn = max(0, total_gt - idtp)
    idf1 = (2 * idtp / (2 * idtp + idfp + idfn)) if (2 * idtp + idfp + idfn) > 0 else float('nan')

    total_ms = float(timing_df['total_ms'].sum()) if not timing_df.empty else 0.0
    fps = (len(timing_df) / (total_ms / 1000.0)) if total_ms > 0 else float('nan')
    mota = 1.0 - ((fn + fp + idsw) / total_gt) if total_gt > 0 else float('nan')

    metrics = {
        'frames': len(gt_by_frame),
        'gt_detections': total_gt,
        'pred_detections': total_pred,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'IDSW_proxy': idsw,
        'precision': tp / (tp + fp) if (tp + fp) > 0 else float('nan'),
        'recall': tp / (tp + fn) if (tp + fn) > 0 else float('nan'),
        'MOTA_proxy': mota,
        'IDF1_proxy': idf1,
        'count_MAE': mean_absolute_error(count_errors),
        'count_RMSE': root_mean_squared_error(count_errors),
        'FPS': fps,
        'infer_ms_avg': float(timing_df['infer_ms'].mean()) if not timing_df.empty else float('nan'),
        'track_ms_avg': float(timing_df['track_ms'].mean()) if not timing_df.empty else float('nan'),
        'total_ms_avg': float(timing_df['total_ms'].mean()) if not timing_df.empty else float('nan'),
        'infer_ms_p95': float(timing_df['infer_ms'].quantile(0.95)) if not timing_df.empty else float('nan'),
        'total_ms_p95': float(timing_df['total_ms'].quantile(0.95)) if not timing_df.empty else float('nan'),
    }

    per_class_rows = []
    for class_name, errors in per_class_errors.items():
        per_class_rows.append({
            'class_name': class_name,
            'MAE': mean_absolute_error(errors),
            'RMSE': root_mean_squared_error(errors),
        })

    return pd.DataFrame([metrics]), pd.DataFrame(per_class_rows).sort_values('class_name')


## 6. Chạy đánh giá

Có thể chỉnh các biến dưới đây trước khi chạy:

- `SELECT_SEQUENCE=None`: tự chọn sequence có nhiều frame nhất.
- `MAX_FRAMES=300`: giới hạn số frame để chạy nhanh hơn.
- `PSEUDO_GT_IOU=0.30`: ngưỡng nối bbox label thành pseudo ID.
- `EVAL_IOU=0.50`: ngưỡng match giữa prediction và pseudo GT khi tính metric.


In [ ]:
SELECT_SEQUENCE = None  # ví dụ: 'video1', 'video2', ...; None = tự chọn sequence dài nhất
MAX_FRAMES = 300
PSEUDO_GT_IOU = 0.30
PSEUDO_GT_MAX_LOST = 2
EVAL_IOU = 0.50
RESIZE_MAX_DIM = settings.stream_max_dim
FRAME_RATE = 30

records = load_sequence_records(SELECT_SEQUENCE, max_frames=MAX_FRAMES)
records = build_pseudo_gt(records, iou_thr=PSEUDO_GT_IOU, max_lost=PSEUDO_GT_MAX_LOST)

gt_by_frame, pred_by_frame, timing_df, predictions_df = run_yolo_bytetrack(
    records,
    resize_max_dim=RESIZE_MAX_DIM,
    frame_rate=FRAME_RATE,
)

metrics_df, per_class_df = evaluate_tracking(
    gt_by_frame,
    pred_by_frame,
    timing_df,
    iou_thr=EVAL_IOU,
)

metrics_path = OUTPUT_DIR / 'bytetrack_metrics_summary.csv'
timing_path = OUTPUT_DIR / 'bytetrack_timing.csv'
pred_path = OUTPUT_DIR / 'bytetrack_predictions.csv'

metrics_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
timing_df.to_csv(timing_path, index=False, encoding='utf-8-sig')
predictions_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

print('Saved:', metrics_path)
print('Saved:', timing_path)
print('Saved:', pred_path)

display(metrics_df.T.rename(columns={0: 'value'}))
display(per_class_df)


## 7. Biểu đồ nhanh

Cell này vẽ sai số số lượng theo frame và thời gian xử lý. Nếu môi trường không có `matplotlib`, cell sẽ bỏ qua phần vẽ.


In [ ]:
if plt is None:
    print('matplotlib is not available')
else:
    plot_df = timing_df.copy()
    plot_df['count_error'] = plot_df['pred_count'] - plot_df['gt_count']

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(plot_df['frame_idx'], plot_df['gt_count'], label='GT count')
    axes[0].plot(plot_df['frame_idx'], plot_df['pred_count'], label='Pred count')
    axes[0].set_ylabel('Vehicles')
    axes[0].legend()

    axes[1].plot(plot_df['frame_idx'], plot_df['count_error'], color='tab:red')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].set_ylabel('Pred - GT')

    axes[2].plot(plot_df['frame_idx'], plot_df['infer_ms'], label='infer_ms')
    axes[2].plot(plot_df['frame_idx'], plot_df['track_ms'], label='track_ms')
    axes[2].plot(plot_df['frame_idx'], plot_df['total_ms'], label='total_ms')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Milliseconds')
    axes[2].legend()

    fig.tight_layout()
    plt.show()


## 8. Kiểm tra trực quan một vài frame

Màu xanh lá: pseudo GT từ label YOLO. Màu đỏ: prediction sau ByteTrack. Nhãn prediction có dạng `track_id:class_name`.


In [ ]:
def draw_box(image: np.ndarray, box: dict, color: tuple[int, int, int], label: str, thickness: int = 2):
    x1, y1, x2, y2 = int(box['x1']), int(box['y1']), int(box['x2']), int(box['y2'])
    cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)
    cv2.putText(image, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

def visualize_frame(record: dict, resize_max_dim: int | None = RESIZE_MAX_DIM):
    image = cv2.imread(str(record['image_path']))
    if image is None:
        raise FileNotFoundError(record['image_path'])
    if resize_max_dim:
        image = resize_to_max(image, resize_max_dim)
    frame_idx = int(record['frame_idx'])
    canvas = image.copy()
    for gt in gt_by_frame.get(frame_idx, []):
        draw_box(canvas, gt, (0, 220, 0), f"GT {gt['pseudo_id']}:{gt['class_name']}")
    for pred in pred_by_frame.get(frame_idx, []):
        track_id = pred.get('track_id', pred.get('object_id', '?'))
        draw_box(canvas, pred, (0, 0, 255), f"{track_id}:{pred.get('class_name', '')}")
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

if plt is None:
    print('matplotlib is not available')
else:
    sample_positions = np.linspace(0, max(len(records) - 1, 0), num=min(4, len(records)), dtype=int)
    fig, axes = plt.subplots(len(sample_positions), 1, figsize=(14, 5 * len(sample_positions)))
    if len(sample_positions) == 1:
        axes = [axes]
    for ax, pos in zip(axes, sample_positions):
        ax.imshow(visualize_frame(records[int(pos)]))
        ax.set_title(f"Frame {records[int(pos)]['frame_idx']}")
        ax.axis('off')
    fig.tight_layout()
    plt.show()


## 9. Khi có ground truth tracking chuẩn

Để tính `MOTA/IDF1` chuẩn, cần file ground truth có ID thật cho từng xe. Format đề xuất:

```csv
frame,id,x,y,w,h,class_name
1,1,120,80,40,60,motor
2,1,124,82,40,60,motor
3,1,129,85,41,61,motor
```

Khi có file này, có thể thay `gt_by_frame` trong notebook bằng ground truth thật thay vì `pseudo_id`. Lúc đó `MOTA_proxy` và `IDF1_proxy` sẽ trở thành metric tracking đáng tin cậy hơn. Nếu cần chuẩn tuyệt đối theo benchmark MOT, nên dùng thêm `motmetrics` hoặc `TrackEval`.
